# DEMO: Window Functions and CTEs in Databricks SQL

Window functions and CTEs are two of the most powerful tools in SQL for analytical work. Together, they let you build complex, readable queries that would otherwise require multiple steps or temporary tables.

In this demo, you’ll learn:
- **Deduplication** with `ROW_NUMBER()` and the `QUALIFY` clause
- **Sliding windows** for rolling averages and cumulative sums
- **CTEs (Common Table Expressions)** to build queries step-by-step

---

In [0]:
%run ./Setup

## Our Data

The setup created two tables:

- **`daily_sales`** — 90 days of daily revenue by region (360 rows). Used for sliding window demos.
- **`sensor_readings`** — IoT sensor data with intentional duplicates (some sensors sent the same reading twice). Used for deduplication demos.

---

In [0]:
%sql
-- Let's first see the duplicate problem in sensor_readings
-- Notice sensor_A at 08:00 and 10:00 appear twice (same timestamp!)
SELECT *
FROM sensor_readings
WHERE sensor_id = 'sensor_A'
ORDER BY reading_timestamp
LIMIT 12;

## Part 1: Deduplication with ROW_NUMBER() and QUALIFY

Duplicate records are one of the most common data quality issues. In SQL, we use `ROW_NUMBER()` to assign a sequence within each group, then keep only the first.

The traditional approach requires a **subquery**:

```sql
SELECT * FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY key ORDER BY timestamp) AS rn
  FROM table
) WHERE rn = 1
```

Databricks offers a cleaner shorthand: the **`QUALIFY`** clause. It filters the results of window functions directly — no subquery needed.

```sql
SELECT *
FROM table
QUALIFY ROW_NUMBER() OVER (PARTITION BY key ORDER BY timestamp) = 1
```

| Part | What it means |
| --- | --- |
| **`ROW_NUMBER() OVER (...)`** | Assign 1, 2, 3... to each row within a group |
| **`PARTITION BY sensor_id, reading_timestamp`** | Restart numbering for each sensor + timestamp combination |
| **`ORDER BY reading_id`** | Within duplicates, the lowest reading_id gets row number 1 |
| **`QUALIFY ... = 1`** | Keep only the first row from each group |

---

In [0]:
%sql
-- ============================================================
-- Deduplication: the traditional subquery approach
-- ============================================================
-- Step 1: assign row numbers within each sensor + timestamp group
-- Step 2: wrap in a subquery and filter for row number = 1

SELECT reading_id, sensor_id, reading_timestamp, temperature, humidity
FROM (
  SELECT 
    *,
    ROW_NUMBER() OVER (
      PARTITION BY sensor_id, reading_timestamp 
      ORDER BY reading_id
    ) AS row_num
  FROM sensor_readings
)
WHERE row_num = 1
ORDER BY sensor_id, reading_timestamp
LIMIT 10;

In [0]:
%sql
-- ============================================================
-- Deduplication: the QUALIFY shorthand (Databricks-specific)
-- ============================================================
-- Same result, but cleaner — no subquery needed!
-- QUALIFY filters on window function results, just like
-- HAVING filters on aggregate results.

SELECT *
FROM sensor_readings
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY sensor_id, reading_timestamp 
  ORDER BY reading_id
) = 1
ORDER BY sensor_id, reading_timestamp
LIMIT 10;

In [0]:
%sql
-- Let's confirm: count before and after dedup
SELECT 
  'Before dedup' AS status,
  COUNT(*) AS row_count
FROM sensor_readings

UNION ALL

SELECT 
  'After dedup' AS status,
  COUNT(*) AS row_count
FROM (
  SELECT *
  FROM sensor_readings
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY sensor_id, reading_timestamp 
    ORDER BY reading_id
  ) = 1
);

## Part 2: Sliding Windows (Rolling Averages)

Sliding windows are essential for time-series analysis — smoothing out daily noise to see trends. In Power BI, you might use a DAX measure with `DATESINPERIOD()` or a custom moving average calculation.

In SQL, we define a **window frame** using `ROWS BETWEEN`:

```sql
AVG(daily_revenue) OVER (
  PARTITION BY region
  ORDER BY sale_date
  ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
)
```

| Part | What it means |
| --- | --- |
| **`ROWS BETWEEN 6 PRECEDING`** | Look back 6 rows before the current row |
| **`AND CURRENT ROW`** | Include the current row (so 7 rows total) |
| **`PARTITION BY region`** | Calculate separately for each region |
| **`ORDER BY sale_date`** | Define what “preceding” means (earlier dates) |

The window “slides” with each row, always looking at the current row plus the 6 before it. This gives you a **7-day rolling average** that smooths out daily volatility.

---

In [0]:
%sql
-- ============================================================
-- Sliding Window: 7-day rolling average of daily revenue
-- ============================================================
-- For each day, calculate the average of that day plus the 6 before it.
-- This smooths out weekend dips and daily noise.

SELECT 
  sale_date,
  region,
  daily_revenue,
  ROUND(AVG(daily_revenue) OVER (
    PARTITION BY region
    ORDER BY sale_date
    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
  ), 2) AS rolling_7day_avg
FROM daily_sales
WHERE region = 'North'
ORDER BY sale_date;

In [0]:
%sql
-- ============================================================
-- Sliding Window: cumulative sum (running total)
-- ============================================================
-- UNBOUNDED PRECEDING = from the very first row to the current row.
-- This gives you a running total that grows every day.

SELECT 
  sale_date,
  region,
  daily_revenue,
  SUM(daily_revenue) OVER (
    PARTITION BY region
    ORDER BY sale_date
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ) AS cumulative_revenue
FROM daily_sales
WHERE region = 'East'
ORDER BY sale_date
LIMIT 15;

In [0]:
%sql
-- ============================================================
-- Sliding Window: comparing each day to the region's average
-- ============================================================
-- An empty OVER() with just PARTITION BY gives the group average.
-- We can then compare each day to its region's overall average.

SELECT 
  sale_date,
  region,
  daily_revenue,
  ROUND(AVG(daily_revenue) OVER (PARTITION BY region), 2) AS region_avg,
  ROUND(daily_revenue - AVG(daily_revenue) OVER (PARTITION BY region), 2) AS vs_average
FROM daily_sales
WHERE sale_date BETWEEN '2024-01-08' AND '2024-01-14'
ORDER BY region, sale_date;

## Part 3: CTEs (Common Table Expressions)

A **CTE** is a named, temporary result set that exists only for the duration of a single query. Think of each CTE block as a **named step** — you can read the logic top to bottom.

```sql
WITH step_one AS (
  SELECT ... FROM ...
),
step_two AS (
  SELECT ... FROM step_one
)
SELECT * FROM step_two;
```

| Feature | What it means |
| --- | --- |
| **`WITH name AS (...)`** | Define a named temporary result |
| **Comma between CTEs** | Chain multiple steps together |
| **Final `SELECT`** | The actual output, usually from the last CTE |

**Why use CTEs?**
- Makes complex queries **readable** (each step has a name)
- Avoids deeply **nested subqueries**
- Each CTE can reference the ones above it
- Nothing is physically stored — it’s all one query

---

In [0]:
%sql
-- ============================================================
-- CTE: Multi-step analysis in one readable query
-- ============================================================
-- This query does 4 things, each as a named step:
-- 1. Calculate weekly revenue per region
-- 2. Add the previous week's revenue (LAG)
-- 3. Calculate week-over-week growth
-- 4. Classify performance

WITH weekly_revenue AS (
  -- Step 1: Aggregate daily data into weekly totals
  SELECT 
    DATE_TRUNC('week', sale_date) AS week_start,
    region,
    SUM(daily_revenue) AS weekly_revenue,
    SUM(order_count) AS weekly_orders
  FROM daily_sales
  GROUP BY DATE_TRUNC('week', sale_date), region
),

with_prior_week AS (
  -- Step 2: Add the previous week's revenue using LAG
  SELECT 
    *,
    LAG(weekly_revenue) OVER (
      PARTITION BY region 
      ORDER BY week_start
    ) AS prior_week_revenue
  FROM weekly_revenue
),

with_growth AS (
  -- Step 3: Calculate week-over-week growth percentage
  SELECT 
    *,
    ROUND(
      (weekly_revenue - prior_week_revenue) / NULLIF(prior_week_revenue, 0) * 100, 1
    ) AS wow_growth_pct
  FROM with_prior_week
)

-- Step 4: Final output with performance classification
SELECT 
  week_start,
  region,
  ROUND(weekly_revenue, 0) AS weekly_revenue,
  ROUND(prior_week_revenue, 0) AS prior_week,
  wow_growth_pct,
  CASE 
    WHEN wow_growth_pct >= 10 THEN 'Strong Growth'
    WHEN wow_growth_pct >= 0 THEN 'Stable'
    WHEN wow_growth_pct >= -10 THEN 'Slight Decline'
    ELSE 'Significant Decline'
  END AS performance
FROM with_growth
WHERE wow_growth_pct IS NOT NULL
ORDER BY week_start, region;

In [0]:
%sql
-- ============================================================
-- CTE + Window Function: Top region each week
-- ============================================================
-- CTEs and window functions are a powerful combination.
-- Here we rank regions by revenue each week and pick the winner.

WITH weekly_totals AS (
  SELECT 
    DATE_TRUNC('week', sale_date) AS week_start,
    region,
    SUM(daily_revenue) AS weekly_revenue
  FROM daily_sales
  GROUP BY DATE_TRUNC('week', sale_date), region
),

ranked AS (
  SELECT 
    *,
    RANK() OVER (PARTITION BY week_start ORDER BY weekly_revenue DESC) AS revenue_rank
  FROM weekly_totals
)

SELECT 
  week_start,
  region AS top_region,
  ROUND(weekly_revenue, 0) AS revenue
FROM ranked
WHERE revenue_rank = 1
ORDER BY week_start;

---

## Summary

| Technique | Use Case |
| --- | --- |
| **`ROW_NUMBER()` + `QUALIFY`** | Remove duplicate records |
| **`ROWS BETWEEN n PRECEDING AND CURRENT ROW`** | Rolling averages, sliding calculations |
| **`ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`** | Running totals, cumulative sums | 
| **`AVG(...) OVER (PARTITION BY ...)`** | Compare each row to its group average |
| **`WITH ... AS (...)`** (CTE) | Break complex logic into readable named steps |

**The key takeaway:** CTEs let you think about a problem in stages. Combined with window functions, you can build sophisticated analytical queries that are readable, debuggable, and self-documenting.